In [50]:
# 1 — Config
source_data_path = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11_samples"
save_csv_path = "/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv"
v11_mini_save_path = "/home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini"

input_csv_name = "GAID_Dataset_v9_full_Train.csv"
input_csv_path = f"{save_csv_path}/{input_csv_name}"
output_csv_path = f"{v11_mini_save_path}/manifest.csv"

# Sampling config
n_fake = 5_000    # label == 0
n_real = 10_000   # label == 1
seed = 42
dry_run = False   # set True to preview counts without copying

In [51]:
# 2 — Imports & helpers
import os
import csv
import shutil
from pathlib import Path
from tqdm.auto import tqdm


def build_filename_index(root: Path) -> dict[str, list[Path]]:
    """Walk *root* and return {basename: [full_path, ...]}."""
    idx: dict[str, list[Path]] = {}
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            idx.setdefault(fname, []).append(Path(dirpath) / fname)
    return idx


print("helpers ready")

helpers ready


In [52]:
# 3 — Add `filename` column to input CSV (basename of image_path) and save back
import pandas as pd

input_csv = Path(input_csv_path)
assert input_csv.is_file(), f"Input CSV not found: {input_csv}"

df_in = pd.read_csv(input_csv)
assert {"image_path", "labels"}.issubset(df_in.columns), \
    "Input CSV must contain 'image_path' and 'labels' columns."

df_in["labels"] = pd.to_numeric(df_in["labels"], errors="coerce").astype("Int64")
df_in["filename"] = df_in["image_path"].astype(str).map(lambda p: os.path.basename(p))

df_in.to_csv(input_csv, index=False)
print(f"Wrote {len(df_in):,} rows back to {input_csv}")
print(f"Columns: {list(df_in.columns)}")
df_in.head(5)

Wrote 704,779 rows back to /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v9_full_Train.csv
Columns: ['image_path', 'labels', 'directory', 'filename']


,image_path,labels,directory,filename
0,/Real/Shutterstock_Real/real/0e35d3c709ef4123b...,1,NaN,0e35d3c709ef4123bb7b8cff23c885a4.jpg
1,/Real/tristanzhang32_Real/real/12905.jpg,1,NaN,12905.jpg
2,/Fake/pixelpulse_1800/fake/1085.jpg,0,NaN,1085.jpg
3,/Real/ulid25k/real/image_05650.png,1,NaN,image_05650.png
4,/Fake/Shutterstock_Fake/fake/d3c613a6d9574f55b...,0,NaN,d3c613a6d9574f55b3854643c5ca2374.jpg


In [53]:
# 4 — Index v11_samples, sample (5k fake + 10k real), copy matches, write manifest
source_root = Path(source_data_path).resolve()
dest_root = Path(v11_mini_save_path).resolve()
output_csv = Path(output_csv_path)

assert source_root.is_dir(), f"Source dir not found: {source_root}"
dest_root.mkdir(parents=True, exist_ok=True)

print(f"Indexing {source_root} …")
filename_index = build_filename_index(source_root)
print(f"  Indexed {len(filename_index):,} unique filenames")
print(f"  Filenames appearing in >1 location: "
      f"{sum(1 for v in filename_index.values() if len(v) > 1):,}")

fake_pool = df_in[df_in["labels"] == 0]
real_pool = df_in[df_in["labels"] == 1]
print(f"Pool sizes — fake/0: {len(fake_pool):,}   real/1: {len(real_pool):,}")
assert len(fake_pool) >= n_fake, f"Only {len(fake_pool)} fake rows, need {n_fake}"
assert len(real_pool) >= n_real, f"Only {len(real_pool)} real rows, need {n_real}"

fake_sample = fake_pool.sample(n=n_fake, random_state=seed)
real_sample = real_pool.sample(n=n_real, random_state=seed)
sampled = pd.concat([fake_sample, real_sample], ignore_index=True)
sampled = sampled.sample(frac=1.0, random_state=seed).reset_index(drop=True)
print(f"Sampled {len(sampled):,} rows  (fake={n_fake:,}, real={n_real:,}, seed={seed})")

rows_out: list[dict[str, str]] = []
missing: list[tuple[str, int]] = []
duplicates: list[str] = []

for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc="Matching & copying"):
    label = int(row["labels"]) if pd.notna(row["labels"]) else -1
    fname = str(row["filename"])

    matches = filename_index.get(fname, [])
    if not matches:
        missing.append((fname, label))
        continue
    if len(matches) > 1:
        duplicates.append(fname)

    src = matches[0]
    rel_from_source = src.relative_to(source_root).as_posix()
    label_dir = "real" if label == 1 else "fake" if label == 0 else "unknown"
    rel_key = f"{label_dir}/{rel_from_source}"
    dest_file = dest_root / rel_key

    if not dry_run:
        dest_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest_file)

    rows_out.append({
        "image_path": str(dest_file),
        "labels": str(label),
        "directory": str(dest_file.parent),
        "filename": fname,
    })

if not dry_run:
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with output_csv.open("w", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(
            f_out, fieldnames=["image_path", "labels", "directory", "filename"]
        )
        w.writeheader()
        w.writerows(rows_out)

action = "Would copy" if dry_run else "Copied"
miss_fake = sum(1 for _, lbl in missing if lbl == 0)
miss_real = sum(1 for _, lbl in missing if lbl == 1)
print(
    f"\n{action} {len(rows_out):,} file(s) → {dest_root}\n"
    f"  Missing source:  {len(missing):,}  (fake={miss_fake:,}, real={miss_real:,})\n"
    f"  Duplicate names: {len(duplicates):,}\n"
    f"  Output CSV:      {output_csv} ({len(rows_out):,} rows)"
)

if missing[:5]:
    print("\nFirst missing filenames:")
    for name, lbl in missing[:5]:
        print(f"  - [{lbl}] {name}")

Indexing /home/taiaburrahman/dataset_manager_pro/data/processed/v11_samples …
  Indexed 867,179 unique filenames
  Filenames appearing in >1 location: 15,897
Pool sizes — fake/0: 361,715   real/1: 343,064
Sampled 15,000 rows  (fake=5,000, real=10,000, seed=42)


Matching & copying: 100%|██████████| 15000/15000 [00:52<00:00, 285.84it/s]



Copied 14,852 file(s) → /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini
  Missing source:  148  (fake=148, real=0)
  Duplicate names: 792
  Output CSV:      /home/taiaburrahman/dataset_manager_pro/data/processed/v11_mini/manifest.csv (14,852 rows)

First missing filenames:
  - [0] Ie1g3ET4kKFZY49X-generated_image.jpg
  - [0] image_4992.jpg
  - [0] JYCd3KEQJPPY9pW6-generated_image.jpg
  - [0] YZffQgeUtzrg6MzH-generated_image.jpg
  - [0] image_3020.jpg


In [54]:
# 5 — Preview the new manifest
import pandas as pd

df = pd.read_csv(output_csv_path)
print(f"Rows: {len(df):,}   Columns: {list(df.columns)}")
print(f"Label counts:\n{df['labels'].value_counts(dropna=False).to_string()}")
df.head(10)

Rows: 14,852   Columns: ['image_path', 'labels', 'directory', 'filename']
Label counts:
labels
1    10000
0     4852


,image_path,labels,directory,filename
0,/home/taiaburrahman/dataset_manager_pro/data/p...,1,/home/taiaburrahman/dataset_manager_pro/data/p...,0d3ba4db-630d-4856-8d05-c4e282a42330.jpg
1,/home/taiaburrahman/dataset_manager_pro/data/p...,1,/home/taiaburrahman/dataset_manager_pro/data/p...,2b82347d48ea4e36a9a2ca5ec591dab4.jpg
2,/home/taiaburrahman/dataset_manager_pro/data/p...,1,/home/taiaburrahman/dataset_manager_pro/data/p...,25a87be1-a2bb-45b5-93bf-dd87daebda71.jpg
3,/home/taiaburrahman/dataset_manager_pro/data/p...,0,/home/taiaburrahman/dataset_manager_pro/data/p...,f0d748f0-ebce-43b8-b045-a9014e8e1b00.jpg
4,/home/taiaburrahman/dataset_manager_pro/data/p...,1,/home/taiaburrahman/dataset_manager_pro/data/p...,19c57e8b-f495-4dd8-ac4a-48ec7a027414.jpg
5,/home/taiaburrahman/dataset_manager_pro/data/p...,1,/home/taiaburrahman/dataset_manager_pro/data/p...,cfddf1e7-d40b-43e7-ad5b-5d6ecb894fbe.jpg
6,/home/taiaburrahman/dataset_manager_pro/data/p...,0,/home/taiaburrahman/dataset_manager_pro/data/p...,b55e0c87fb004b72a92c2523ece70284.jpg
7,/home/taiaburrahman/dataset_manager_pro/data/p...,0,/home/taiaburrahman/dataset_manager_pro/data/p...,92022cb6-0ede-4485-8313-fd8dd183ea09.jpg
8,/home/taiaburrahman/dataset_manager_pro/data/p...,0,/home/taiaburrahman/dataset_manager_pro/data/p...,Gbx8_2BWkAYDEd-.jpg
9,/home/taiaburrahman/dataset_manager_pro/data/p...,0,/home/taiaburrahman/dataset_manager_pro/data/p...,53d201e3e12843fd9d9f7512243943dd.jpg


In [57]:
for i in range(len(df)):
    output_csv_path = df.iloc[i]['image_path']
    file_exists = os.path.exists(output_csv_path)
    if not file_exists:
        print(f"File does not exist: {output_csv_path}")
